# Config

In [2]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [3]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [4]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json
import numpy as np

# Preparación de texto

In [72]:
# 1) Cargar datos
path = "/tmp/ocde/new_data"

#Cargar titulo, keys, etc.
filePATH = os.path.join(path, "old_data.xlsx")
df_data = pd.read_excel(filePATH, usecols=["Código VRID", "Título", "Keywords", "Resumen", "Depto Persona",
                                           "Facultad del Proyecto"])

#Cargar desafíos
filePATH = os.path.join(path, "new_data.xlsx")
#Pagina 0
df_ocde0 = pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Categoria Disciplina", 
                                             "Sub Area Disciplina", "Tipo Área Disciplina"], sheet_name=2)
#Pagina 1
df_ocde1 =  pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Categoria Disciplina", 
                                             "Sub Area Disciplina", "Tipo Área Disciplina"], sheet_name=3)

df_labels = pd.concat([df_ocde0, df_ocde1])


#Codes unique
codes_unique = df_labels["Código VRID"].astype(str).unique()
print(codes_unique.shape)
print(df_data.shape, df_labels.shape)
print(np.unique(df_data["Código VRID"].astype(str)).shape)
print(np.unique(df_labels["Código VRID"].astype(str)).shape)

(565,)
(1083, 6) (645, 4)
(1083,)
(565,)


In [73]:
df_data["Código VRID"] = df_data["Código VRID"].astype(str)
df_labels["Código VRID"] = df_labels["Código VRID"].astype(str)


# Merge en base a "Código VRID"
df_merged = pd.merge(
    df_data,
    df_labels,
    on="Código VRID",       # columna clave
    how="inner"             # inner = solo los que coinciden en ambos
)
#Eliminar dupliados
df_merged = df_merged.drop_duplicates(subset=["Código VRID"], keep="first")
print(df_merged.shape)                                                                                                                                
#print(np.unique(df_merged["Código VRID"].astype(str)).shape)
print(df_merged["Categoria Disciplina"].value_counts())

#save dataframe
savepath=os.path.join(path, "data_concatenada.csv")
df_merged.to_csv(savepath, index=False, encoding="utf-8-sig")

(409, 9)
CIENCIAS NATURALES                        202
 INGENIERÍA, TECNOLOGÍA                    72
CIENCIAS SOCIALES                          57
CIENCIAS AGRÍCOLAS                         31
HUMANIDADES                                26
CIENCIAS MÉDICAS, CIENCIAS DE LA SALUD     21
Name: Categoria Disciplina, dtype: int64


# 1) Preprocesamiento de los datos


In [49]:
# 1) Cargar datos
path = "/tmp/ocde/new_data"
filePATH = os.path.join(path, "data_concatenada.csv")
df = pd.read_csv(filePATH)

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Depto Persona": "Depto_Persona_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad", 
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

# 2) Traducción del texto

In [50]:
#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()

#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

In [53]:
df.head()

,Código VRID,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Categoria Disciplina,Sub Area Disciplina,Tipo Área Disciplina,Titulo_trad,Resumen_trad,keywords_trad,Depto_Persona_trad,Facultad_del_Proyecto_trad,Español
0,11170959,PRE-AND POST-NATAL EXPRESSION OF SLIT2 AND ITS...,"NEUROBLAST MIGRATION, SLIT2, LATERAL VENTRICLE...",IN THE SUBVENTRICULAR ZONE (SVZ) OF THE ADULT ...,FACULTAD DE CIENCIAS BIOLÓGICAS,DEPARTAMENTO DE FARMACOLOGÍA,CIENCIAS NATURALES,BIOLOGÍA,OCDE,pre-and post-natal expression of slit2 and its...,in the subventricular zone (svz) of the adult ...,"neuroblast migration, slit2, lateral ventricle...",department of pharmacology,Faculty of Life Sciences,False
1,11181200,"""DEVELOPMENT AND VALIDATION OF AN ANALYTICAL P...","CHROMATOGRAPHY-MASS SPECTROMETRY, UNTARGETED M...",_x000D_\nACCORDING TO THE INTERNATIONAL AGENCY...,FACULTAD DE FARMACIA,DEPARTAMENTO DE ANÁLISIS INSTRUMENTAL,CIENCIAS NATURALES,QUÍMICA,OCDE,"""development and validation of an analytical p...",according to the international agency for rese...,"chromatography-mass spectrometry, untargeted m...",departamento de análisis instrumental,faculty of pharmacy,False
2,11190637,TRANSGENERATIONAL AND WITHIN-GENERATION PLASTI...,NaN,NaN,FACULTAD DE CIENCIAS NATURALES Y OCEANOGRÁFICAS,DEPARTAMENTO DE ZOOLOGÍA,"CIENCIAS MÉDICAS, CIENCIAS DE LA SALUD",MEDICINA BÁSICA,OCDE,transgenerational and within-generation plasti...,,,zoology department,Faculty of Natural and Oceanographic Sciences,False
3,11190698,: DESIGN AND SYNTHESIS OF ENANTIOPURE HYDROISO...,"HYDROISOQUINOLINES, ORGANOCATALYSIS, TANDEM RE...","IN THE LAST DECADES, THE SYNTHESIS OF NATURAL ...",FACULTAD DE CIENCIAS QUÍMICAS,DEPARTAMENTO DE QUÍMICA ORGÁNICA,CIENCIAS NATURALES,OTRAS CIENCIAS NATURALES,OCDE,design and synthesis of enantiopure hydroisoqu...,"in the last decades, the synthesis of natural ...","hydroisoquinolines, organocatalysis, tandem re...",Department of Organic Chemistry,Faculty of Chemical Sciences,False
4,11190906,CRYPTIC SPECIATION IN THE SOUTHERN OCEAN: INTE...,"SOUTHERN OCEAN, CRYPTIC SPECIATION, MARINE INV...","SINCE THE ISOLATION OF ANTARCTICA, THE INTERAC...",FACULTAD DE CIENCIAS NATURALES Y OCEANOGRÁFICAS,DEPARTAMENTO DE ZOOLOGÍA,CIENCIAS NATURALES,BIOLOGÍA,OCDE,cryptic speciation in the southern ocean: inte...,"since the isolation of antarctica, the interac...","southern ocean, cryptic speciation, marine inv...",zoology department,Faculty of Natural and Oceanographic Sciences,False


# 3) Split dataset

In [71]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from utils.dataset import to_serializable
import json

path = "/tmp/ocde/new_data"
filepath=os.path.join(path, "data_translated.csv")
df = pd.read_csv(filepath)


In [72]:
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Categoria Disciplina"].astype(str))

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "new_data_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)

Test size: 82
Fold 0 - Val size: 109


# 4) TF-ID feature extractor 

## Train

In [121]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/ocde/new_data"
filepath=os.path.join(path, "new_data_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "data_translated.csv")
df = pd.read_csv(filepath)

In [122]:
df["Categoria Disciplina"].shape
df["Categoria Disciplina"].value_counts()

elements = ["CIENCIAS NATURALES", "INGENIERÍA, TECNOLOGÍA", "CIENCIAS SOCIALES", 
             "CIENCIAS AGRÍCOLAS", "HUMANIDADES", "CIENCIAS MÉDICAS, CIENCIAS DE LA SALUD"]

# Clean spaces
df["Categoria Disciplina"] = df["Categoria Disciplina"].str.strip()

# Create one-hot encoded columns
for cat in elements:
    df[cat] = (df["Categoria Disciplina"] == cat).astype(int)


In [123]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_vectors
from utils.dataset import gen_dataset_select_cols, binarize_labels
import numpy as np


#Lectura de codigos 
codes_test = dataset_index["Test"]
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
col_test = "CIENCIAS MÉDICAS, CIENCIAS DE LA SALUD"
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                test_col=col_test)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols, 
                                                     test_col=col_test)
df_decode = df_train[["idx", "Código VRID"]]

#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

#Encode labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(y_train.shape, y_test.shape)


(327, 11947) (82, 11947)
(327,) (82,)


In [124]:
from collections import Counter
print("Train label distribution:", Counter(y_train))
print("Test label distribution:", Counter(y_test))

Train label distribution: Counter({0: 310, 1: 17})
Test label distribution: Counter({0: 78, 1: 4})


In [125]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model#, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'

#from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(random_state=7, shuffle=True)
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=skf, 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({0: 310, 1: 17})
📊 test: Counter({0: 78, 1: 4})
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbf

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.57, 'std_test_score': 0.1}
RandomForestClassifier: {'mean_test_score': 0.49, 'std_test_score': 0.0}
XGBClassifier: {'mean_test_score': 0.57, 'std_test_score': 0.1}
SVC: {'mean_test_score': 0.58, 'std_test_score': 0.11}


In [126]:
from utils.mlflow import eval_model
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.9512195121951219, 'f1_macro': 0.4875, 'cm': array([[78,  0],
       [ 4,  0]]), 'precision': 0.0, 'recall': 0.0, 'f1_es': 0.8861538461538463, 'f1_en': 0.9467532467532467, 'cm_es': array([[24,  0],
       [ 2,  0]]), 'cm_en': array([[54,  0],
       [ 2,  0]])}
RandomForestClassifier
{'accuracy': 0.9512195121951219, 'f1_macro': 0.4875, 'cm': array([[78,  0],
       [ 4,  0]]), 'precision': 0.0, 'recall': 0.0, 'f1_es': 0.8861538461538463, 'f1_en': 0.9467532467532467, 'cm_es': array([[24,  0],
       [ 2,  0]]), 'cm_en': array([[54,  0],
       [ 2,  0]])}
XGBClassifier
{'accuracy': 0.9390243902439024, 'f1_macro': 0.48427672955974843, 'cm': array([[77,  1],
       [ 4,  0]]), 'precision': 0.0, 'recall': 0.0, 'f1_es': 0.8861538461538463, 'f1_en': 0.937745740498034, 'cm_es': array([[24,  0],
       [ 2,  0]]), 'cm_en': array([[53,  1],
       [ 2,  0]])}
SVC
{'accuracy': 0.9512195121951219, 'f1_macro': 0.4875, 'cm': array([[78,  0],
       [ 4,  0]]), 'prec

## Save

In [127]:
from utils.mlflow import mlflow_ckeckpoint

# Guardar en MLflow
exp_info = {
    'exp_name': "Bayesiansearchcv_OCDE_all_results",
}

extra_parms = {
    "vectorization_model": "TF-IDF",
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols,
    "ocde area": col_test,
}

extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, mode="server", mode_classification="binary")

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/22 02:40:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/22/runs/be9559bab1384c1cac70e929978319f9
🧪 View experiment at: http://mlflow-server:5000/#/experiments/22
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/22 02:40:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/22/runs/a208bb48a53749a09892df3abc3dbc4b
🧪 View experiment at: http://mlflow-server:5000/#/experiments/22
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/22 02:40:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/22/runs/1fac1f2fd31b4be7b89566a2460dbbdb
🧪 View experiment at: http://mlflow-server:5000/#/experiments/22
📝 Registrando modelo en MLflow: SVC


2025/09/22 02:40:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/22/runs/506597455f6c476488c27b467c7a22f9
🧪 View experiment at: http://mlflow-server:5000/#/experiments/22
